# 01. EDA & Log Preprocessing

본 노트북은 `LogSentinel` 프로젝트의 1~2단계에 해당하는 **로그 데이터 탐색(EDA)** 및 **정규식 기반 텍스트 정제(Parsing)**, **TF-IDF 벡터화** 과정의 작동성을 검증하는 공간입니다.

In [2]:
%load_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 프로젝트 루트 경로를 sys.path에 추가하여 src 패키지를 찾을 수 있도록 함
sys.path.append(os.path.abspath(os.path.join("..")))

# 작성한 모듈 임포트
from src.parser import LogParser

# 한글 폰트 설정 (추가됨)
import platform

if platform.system() == "Darwin":  # Mac
    plt.rcParams["font.family"] = "AppleGothic"
elif platform.system() == "Windows":  # Windows
    plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

## 1. 데이터 로드 및 정상/장애 통계 확인
BGL 로그의 1열이 `-`이면 정상, 그 외의 태그이면 이상(Anomaly)을 의미합니다.

In [3]:
# 경로 설정 (T7 외장 SSD 우선, 없으면 로컬 fallback)
SSD_PATH = "/Volumes/T7/LogSentinel_Data/raw/"
# CWD가 notebooks 디렉토리일 경우를 고려하여 프로젝트 루트 기준으로 상대경로를 절대경로로 변환합니다.
LOCAL_PATH = os.path.abspath(os.path.join("..", "data", "raw"))

def get_data_path(filename):
    ssd_file = os.path.join(SSD_PATH, filename)
    local_file = os.path.join(LOCAL_PATH, filename)
    if os.path.exists(ssd_file):
        return ssd_file
    return local_file

log_file_path = get_data_path("BGL_2k.log")
print(f"Using log file: {log_file_path}")

Using log file: /Users/soromiso/Desktop/Dev/soromiso/LogSentinel/data/raw/BGL_2k.log


In [4]:
# 로그 데이터 파싱하여 리스트업
data = []
with open(log_file_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) < 10: 
            continue
        label = parts[0]
        is_anomaly = 1 if label != "-" else 0
        message = " ".join(parts[9:])
        data.append({"label_raw": label, "is_anomaly": is_anomaly, "raw_message": message})

df = pd.DataFrame(data)
print(f"Total parsed logs: {len(df)}")
print(df["is_anomaly"].value_counts(normalize=True))

Total parsed logs: 2000
is_anomaly
0    0.9285
1    0.0715
Name: proportion, dtype: float64


In [5]:
df

,label_raw,is_anomaly,raw_message
0,-,0,instruction cache parity error corrected
1,-,0,instruction cache parity error corrected
2,-,0,instruction cache parity error corrected
3,-,0,instruction cache parity error corrected
4,-,0,63543 double-hummer alignment exceptions
...,...,...,...
1995,-,0,instruction cache parity error corrected
1996,-,0,instruction cache parity error corrected
1997,-,0,instruction cache parity error corrected
1998,-,0,instruction cache parity error corrected


## 2. 정규식 파서 전처리 테스트

In [6]:
parser = LogParser(max_features=500)
sample_log1 = df.iloc[1]["raw_message"]
sample_log2 = df.iloc[2]["raw_message"]
sample_log9 = df.iloc[9]["raw_message"]

print(f'sample_log1: {sample_log1}')
print(f'sample_log2: {sample_log2}')
print(f'sample_log9: {sample_log9}')

sample_log1: instruction cache parity error corrected
sample_log2: instruction cache parity error corrected
sample_log9: ciod: failed to read message prefix on control stream (CioStream socket to 172.16.96.116:33370


In [7]:
# 샘플 클리닝 테스트
sample_log_09 = df.iloc[9]["raw_message"]
sample_log_10 = df.iloc[10]["raw_message"]
sample_log_21 = df.iloc[21]["raw_message"]
sample_log_32 = df.iloc[32]["raw_message"]

print("Sample Raw[9]     :", sample_log_09)
print("Sample Cleaned[9] :", parser.clean_log(sample_log_09))

print("Sample Raw[10]    :", sample_log_10)
print("Sample Cleaned[10]:", parser.clean_log(sample_log_10))

print("Sample Raw[21]    :", sample_log_21)
print("Sample Cleaned[21]:", parser.clean_log(sample_log_21))

print("Sample Raw[32]    :", sample_log_32)
print("Sample Cleaned[32]:", parser.clean_log(sample_log_32))

Sample Raw[9]     : ciod: failed to read message prefix on control stream (CioStream socket to 172.16.96.116:33370
Sample Cleaned[9] : ciod: failed to read message prefix on control stream (ciostream socket to [ip] : [num]
Sample Raw[10]    : CE sym 20, at 0x1438f9e0, mask 0x40
Sample Cleaned[10]: ce sym [num] , at [hex] , mask [hex]
Sample Raw[21]    : generating core.201
Sample Cleaned[21]: generating core. [num]
Sample Raw[32]    : generating core.6471
Sample Cleaned[32]: generating core. [num]


In [8]:
# 전체 데이터 정제 적용
df["cleaned_message"] = df["raw_message"].apply(parser.clean_log)
print(df[["raw_message", "cleaned_message"]].head())

                                raw_message  \
0  instruction cache parity error corrected   
1  instruction cache parity error corrected   
2  instruction cache parity error corrected   
3  instruction cache parity error corrected   
4  63543 double-hummer alignment exceptions   

                            cleaned_message  
0  instruction cache parity error corrected  
1  instruction cache parity error corrected  
2  instruction cache parity error corrected  
3  instruction cache parity error corrected  
4  [num] double-hummer alignment exceptions  


## 3. TF-IDF 벡터화 실행

In [9]:
tfidf_matrix = parser.fit_transform(df["cleaned_message"])
print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (2000, 500)


In [10]:
tfidf_matrix[:100] # 99행 까지만 살작 보자

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [11]:
# 경로 설정 및 방어 코딩 (T7 SSD 장착 여부 확인)
SSD_MODELS_PATH = "/Volumes/T7/LogSentinel_Data/models"
LOCAL_MODELS_PATH = os.path.abspath(os.path.join("..", "models"))
VECTORIZER_PKL = 'vectorizer_fr_2k.pkl'

# T7 SSD 디렉토리가 존재하는지 확인
if os.path.exists("/Volumes/T7"):
    models_dir = SSD_MODELS_PATH
else:
    models_dir = LOCAL_MODELS_PATH

os.makedirs(models_dir, exist_ok=True)
vectorizer_path = os.path.join(models_dir, VECTORIZER_PKL)

# 저장 시도 및 예외 처리
try:
    parser.save_vectorizer(vectorizer_path)
    print(f"Vectorizer saved successfully at: {vectorizer_path}")
except Exception as e:
    print(f"Failed to save at {vectorizer_path}. Error: {e}")
    # 로컬 경로로 fallback 저장 시도
    if models_dir != LOCAL_MODELS_PATH:
        print("Attempting to save to local models directory as fallback...")
        os.makedirs(LOCAL_MODELS_PATH, exist_ok=True)
        fallback_path = os.path.join(LOCAL_MODELS_PATH, "vectorizer.pkl")
        try:
            parser.save_vectorizer(fallback_path)
            print(f"Successfully saved to fallback path: {fallback_path}")
        except Exception as fallback_err:
            print(f"Critical: Fallback save failed. Error: {fallback_err}")

Vectorizer saved successfully at: /Users/soromiso/Desktop/Dev/soromiso/LogSentinel/models/vectorizer_fr_2k.pkl
